# CogMem Phase 1 — Paperspace Setup\n\nCollect Q-valued episodes from MemRL on ALFWorld using Paperspace A4000 GPU.

In [4]:
# Cell 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

NVIDIA RTX A4000, 16376 MiB, 16101 MiB


In [1]:
# Cell 2: Install system deps + Ollama
!apt-get update -qq && apt-get install -y -qq zstd cmake build-essential pciutils > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0% 92.5%
>>> Creating ollama user...
>>> Adding ollama user to render group...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
# Cell 3: Start Ollama server in background
import subprocess, time
proc = subprocess.Popen(
    ["ollama", "serve"],
    env={**__import__("os").environ, "OLLAMA_HOST": "0.0.0.0:11434"},
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)
print(f"Ollama PID: {proc.pid}")
!curl -s http://localhost:11434/api/tags | python3 -c "import sys; print(sys.stdin.read()[:100])"

Ollama PID: 768
{"models":[]}


In [ ]:
# Cell 4: Pull models
!ollama pull llama3.2:3b
!ollama pull nomic-embed-text

In [ ]:
# Cell 5: Install ALFWorld + MemRL + ALL dependencies
!pip install alfworld -q && alfworld-download
!cd /notebooks && git clone https://github.com/MemTensor/MemRL 2>/dev/null
# Install MemRL deps (skip torch — keep Paperspace's CUDA version)
!cd /notebooks/MemRL && grep -v "^torch==" requirements.txt | grep -v "^transformers==" | pip install -r /dev/stdin -q && pip install -e . --no-deps -q
!pip install "chonkie==1.2.1" qdrant-client tensorboard tiktoken openai textworld -q
# Pin compatible versions for Paperspace's torch 2.1.1 + CUDA 12.1
!pip install "pydantic>=2.0" "transformers>=4.35,<4.38" -q

In [1]:
# Cell 6: Setup ALFWorld data + few-shot examples
import os, shutil
from pathlib import Path

memrl = Path("/notebooks/MemRL")
alf_data = memrl / "data" / "alfworld"
alf_data.mkdir(parents=True, exist_ok=True)

# Find alfworld cache
cache = Path.home() / ".cache" / "alfworld"
if not cache.exists():
    cache = Path("/root/.cache/alfworld")

# Symlink data
for name in ["json_2.1.1", "logic", "detectors"]:
    src = cache / name
    dst = alf_data / name
    if src.exists() and not dst.exists():
        os.symlink(str(src), str(dst))
        print(f"Linked {name}")

# Copy few-shot examples
src_examples = memrl / "configs" / "alfworld" / "alfworld_examples.json"
dst_examples = alf_data / "alfworld_examples.json"
if src_examples.exists() and not dst_examples.exists():
    shutil.copy2(src_examples, dst_examples)
    print("Copied few-shot examples")

# Verify
!ls -la /notebooks/MemRL/data/alfworld/

total 19
drwxr-xr-x 2 root root     4 Apr  2 14:57 .
drwxr-xr-x 5 root root     3 Apr  2 14:57 ..
-rw-r--r-- 1 root root 17352 Apr  2 14:56 alfworld_examples.json
lrwxrwxrwx 1 root root    31 Apr  2 14:57 detectors -> /root/.cache/alfworld/detectors
lrwxrwxrwx 1 root root    32 Apr  2 14:57 json_2.1.1 -> /root/.cache/alfworld/json_2.1.1
lrwxrwxrwx 1 root root    27 Apr  2 14:57 logic -> /root/.cache/alfworld/logic


In [ ]:
# Cell 7: Create local config for Paperspace (localhost Ollama, A4000 GPU)
config = """
# Paperspace A4000 config for CogMem Phase 1 — llama3.2:3b
llm:
  provider: "openai"
  api_key: "ollama"
  base_url: "http://localhost:11434/v1"
  model: "llama3.2:3b"
  temperature: 0
  max_tokens: 4096

embedding:
  provider: "openai"
  api_key: "ollama"
  base_url: "http://localhost:11434/v1"
  model: "nomic-embed-text"
  max_text_len: 4096

memory:
  build_strategy: "proceduralization"
  retrieve_strategy: "query"
  update_strategy: "adjustment"
  k_retrieve: 5
  max_keywords: 8
  confidence_threshold: 0.0
  memory_confidence: 100.0
  add_similarity_threshold: 0.90
  mos_config_path: "configs/mos_config_final.json"
  user_id: "memrl_user"
  sim_norm_mean: 0.5187
  sim_norm_std: 0.1203

environment:
  alfworld_config_path: "configs/envs/alfworld.yaml"
  alfworld_env_type: "AlfredTWEnv"

experiment:
  random_seed: 42
  enable_value_driven: true
  experiment_name: 'cogmem_phase1_3b'
  mode: 'train'
  num_sections: 1
  batch_size: 8
  dataset_ratio: 0.07
  few_shot_path: 'data/alfworld/alfworld_examples.json'
  max_steps: 20
  bon: 1
  valid_interval: 1
  test_interval: 5
  baseline_mode: none
  baseline_k: 10
  output_dir: "./results"
  save_trajectories: true
  save_memories: true
  ckpt_eval_enabled: false
  ckpt_eval_path: ""
  ckpt_resume_enabled: false
  ckpt_resume_path: ""
  ckpt_resume_epoch: null

rl_config:
  epsilon: 0
  tau: 0.62
  alpha: 0.3
  gamma: 0.0
  q_init_pos: 0
  q_init_neg: 0
  success_reward: 1.0
  failure_reward: -1.0
  topk: 3
  novelty_threshold: 0.85
  recency_boost: 0.0
  reward_merge_gain: 0.1
  q_min_threshold: -10
  weight_sim: 0.5
  weight_q: 0.5
""".strip()

with open("/notebooks/MemRL/configs/rl_alf_config.local.yaml", "w") as f:
    f.write(config)
print("Config written! llama3.2:3b, ~248 games, 1 section, ~30 batches, fits in 6h")

In [ ]:
# Cell 7.5: Test file export — check where files land
!echo "hello from paperspace" > /notebooks/test_export.txt
!pwd
!ls -la /notebooks/test_export.txt
print("Check if test_export.txt appears in Paperspace file browser under /notebooks/")

In [ ]:
# Cell 8: Quick test — verify Ollama + ALFWorld + MemRL all work
!curl -s http://localhost:11434/api/generate -d '{"model":"llama3.2:3b","prompt":"Say OK","stream":false}' | python3 -c "import sys,json; d=json.load(sys.stdin); print('Ollama OK:', d['response'][:20])"
!python3 -c "import alfworld; print('ALFWorld OK')"
!cd /notebooks/MemRL && python3 -c "from memrl.configs.config import MempConfig; cfg = MempConfig.from_yaml('configs/rl_alf_config.local.yaml'); print(f'MemRL OK: {cfg.experiment.experiment_name}, batch={cfg.experiment.batch_size}, sections={cfg.experiment.num_sections}')"

In [ ]:
# Cell 9: RUN Phase 1 — Episode Collection (llama3.2:3b)
# 355 games x 2 sections = 710 episodes, batch_size=8, max_steps=20
# Estimated ~2.5 hours on A4000
!cd /notebooks/MemRL && python3 run/run_alfworld.py --config configs/rl_alf_config.local.yaml

In [ ]:
# Cell 9b: RESUME from the best run (31 batches completed)
import yaml

config_path = "/notebooks/MemRL/configs/rl_alf_config.local.yaml"
with open(config_path) as f:
    cfg = yaml.safe_load(f)

# Point to the 235940 run's local_cache (the one with 31 batches done)
cfg["experiment"]["ckpt_resume_enabled"] = True
cfg["experiment"]["ckpt_resume_path"] = "/notebooks/MemRL/results/alfworld/exp_cogmem_phase1_3b_20260402-235940/local_cache"
cfg["experiment"]["ckpt_resume_epoch"] = 31  # resume from after batch 31

with open(config_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("Config updated! Resuming from batch 31 of exp_cogmem_phase1_3b_20260402-235940")
print("Now re-run Cell 9")

In [ ]:
# Cell 10: After completion — check results and download cube dump
!ls -la /notebooks/MemRL/results/mem_cubes/
!echo "---"
# Find the latest snapshot
!find /notebooks/MemRL/results -name "*.json" -path "*/mem_cubes/*" | head -20

In [ ]:
# Cell 11: Package results for download
!cd /notebooks/MemRL && tar czf /notebooks/cogmem_phase1_results.tar.gz results/mem_cubes/ results/alfworld/ 2>/dev/null; ls -lh /notebooks/cogmem_phase1_results.tar.gz
print("Download cogmem_phase1_results.tar.gz from Paperspace file browser")